# Stage 2: Split Catalog by Chapters

**Purpose**: Separates the monolithic BC3 JSON file into individual chapter files based on ADIF's organizational structure.

**Input**: `data/intermediate/BPA_2024_v2_OEB_mod_utf8.json`  
**Output**: `data/intermediate/{CHAPTER_NAME}/{CHAPTER_NAME}.json`

## What this notebook does

The ADIF catalog is organized into technical chapters, identified by the first character of each item key:

| Code | Chapter |
|------|---------|
| A | ARQUITECTURA (Architecture) |
| C | CONTROL MANDO Y SEÑALIZACIÓN (Control & Signaling) |
| E | ENERGIA (Energy) |
| G | GESTIÓN AMBIENTAL (Environmental Management) |
| O | **OBRA CIVIL** (Civil Works) ← Contains OEB subcategory |
| P | PROTECCIÓN Y SEGURIDAD (Protection & Security) |
| S | SEGURIDAD Y SALUD (Safety & Health) |
| T | TELECOMUNICACIONES (Telecommunications) |
| V | VIA (Railway Track) |

Each chapter is saved as a separate JSON file for independent processing.

In [10]:
import json
import os

def split_chapters(input_file):
    """
    Splits a large JSON price database into separate chapter files.
    Each chapter is determined by the first character of the item code.
    """
    # Read and parse the JSON file
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Dictionary to store items grouped by their chapter code
    chapters = {}
    
    # Iterate through all items in the input JSON
    for key, value in data.items():
        # Skip special entries (%, R_A_I_Z, AU)
        if key.startswith('%') or key.startswith('R_A_I_Z') or key.startswith('AU'):
            continue
        
        # Get chapter code (first character of the item code)
        chapter_code = key[0]
        
        # Initialize chapter dictionary if not exists
        if not chapters.get(chapter_code):
            chapters[chapter_code] = {}
            
        # Add item to its corresponding chapter
        chapters[chapter_code][key] = value
    
    # Create 'chapters' directory next to the input file
    output_dir = os.path.join(os.path.dirname(input_file), 'chapters')
    os.makedirs(output_dir, exist_ok=True)
    
    # Process each chapter
    for code, items in chapters.items():
        # Find the main chapter item (ends with '#')
        main_item = items.get(f"{code}#")
        if not main_item:
            continue
        
        # Get chapter name from the concept field
        chapter_name = main_item['concept']
        
        # Create folder for each chapter
        base_dir = os.path.dirname(input_file)
        chapter_dir = os.path.join(base_dir, chapter_name)
        os.makedirs(chapter_dir, exist_ok=True)
        
        # Save JSON file inside chapter folder
        filename = os.path.join(chapter_dir, f"{chapter_name}.json")
        
        # Save chapter items to a new JSON file
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(items, f, ensure_ascii=False, indent=2)
        
        # Print creation confirmation and item count
        print(f"Created: {filename} ({len(items)} items)")

In [11]:
split_chapters("/work/data/intermediate/BPA_2024_v2_OEB_mod_utf8.json")

Created: /work/data/intermediate/ARQUITECTURA/ARQUITECTURA.json (844 items)
Created: /work/data/intermediate/CONTROL MANDO Y SEÑALIZACIÓN/CONTROL MANDO Y SEÑALIZACIÓN.json (546 items)
Created: /work/data/intermediate/ENERGIA/ENERGIA.json (932 items)
Created: /work/data/intermediate/GESTIÓN AMBIENTAL/GESTIÓN AMBIENTAL.json (66 items)
Created: /work/data/intermediate/PRECIOS BÁSICOS/PRECIOS BÁSICOS.json (23806 items)
Created: /work/data/intermediate/OBRA CIVIL/OBRA CIVIL.json (545 items)
Created: /work/data/intermediate/CONTROL Y PRUEBAS/CONTROL Y PRUEBAS.json (668 items)
Created: /work/data/intermediate/PROTECCIÓN Y SEGURIDAD/PROTECCIÓN Y SEGURIDAD.json (137 items)
Created: /work/data/intermediate/SEGURIDAD Y SALUD/SEGURIDAD Y SALUD.json (182 items)
Created: /work/data/intermediate/TELECOMUNICACIONES/TELECOMUNICACIONES.json (490 items)
Created: /work/data/intermediate/VIA/VIA.json (278 items)
